# **Reinforcement Learning - Deep Q-Learning with CartPole**
Implementasi Deep Q-Network (DQN) untuk memecahkan masalah CartPole dari OpenAI Gym.

In [12]:
pip install --upgrade gym

In [13]:
# Install library yang dibutuhkan
!pip install gym
!pip install tensorflow

In [14]:
import numpy as np
import gym
import tensorflow as tf
from tensorflow import keras
from collections import deque
import random

In [15]:
# Membuat environment CartPole
env = gym.make("CartPole-v1")
input_shape = env.observation_space.shape
n_outputs = env.action_space.n

In [16]:
# Membuat Deep Q-Network
model = keras.models.Sequential([
    keras.layers.Dense(32, activation="elu", input_shape=input_shape),
    keras.layers.Dense(32, activation="elu"),
    keras.layers.Dense(n_outputs)
])

In [17]:
# Policy ε-greedy untuk memilih aksi
def epsilon_greedy_policy(state, epsilon=0):
    if np.random.rand() < epsilon:
        return np.random.randint(n_outputs)
    else:
        Q_values = model.predict(state[np.newaxis], verbose=0)
        return np.argmax(Q_values[0])

In [18]:
# Replay buffer untuk menyimpan pengalaman
replay_buffer = deque(maxlen=2000)

def sample_experiences(batch_size):
    indices = np.random.randint(len(replay_buffer), size=batch_size)
    batch = [replay_buffer[index] for index in indices]
    states, actions, rewards, next_states, dones = [
        np.array([experience[field] for experience in batch]) for field in range(5)
    ]
    return states, actions, rewards, next_states, dones

In [19]:
# Fungsi untuk menjalankan 1 langkah simulasi
def play_one_step(env, state, epsilon):
    action = epsilon_greedy_policy(state, epsilon)
    next_state, reward, done, info = env.step(action)
    replay_buffer.append((state, action, reward, next_state, done))
    return next_state, reward, done

In [20]:
# Fungsi training_step

batch_size = 32
discount_factor = 0.95
optimizer = keras.optimizers.Adam(learning_rate=1e-3)
loss_fn = keras.losses.MeanSquaredError()  # Perbaikan: pakai kelas, bukan fungsi biasa

def training_step():
    experiences = sample_experiences(batch_size)
    states, actions, rewards, next_states, dones = experiences
    next_Q_values = model.predict(next_states, verbose=0)
    max_next_Q_values = np.max(next_Q_values, axis=1)
    target_Q_values = rewards + (1 - dones) * discount_factor * max_next_Q_values
    mask = tf.one_hot(actions, n_outputs)

    with tf.GradientTape() as tape:
        all_Q_values = model(states)
        Q_values = tf.reduce_sum(all_Q_values * mask, axis=1)
        loss = loss_fn(target_Q_values, Q_values)  # ✅ Perbaikan di sini
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

In [22]:
# Training Loop
n_episodes = 500
for episode in range(n_episodes):
    obs, info = env.reset()  # ✅ Versi gym>=0.25
    done = False
    total_reward = 0
    while not done:
        epsilon = max(1 - episode / 400, 0.01)
        action = epsilon_greedy_policy(obs, epsilon)
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        replay_buffer.append((obs, action, reward, next_obs, done))
        obs = next_obs
        total_reward += reward

        if episode >= 50:
            training_step()

    if episode % 20 == 0:
        print(f"Episode {episode}: Total Reward = {total_reward}")

Episode 0: Total Reward = 27.0
Episode 20: Total Reward = 29.0
Episode 40: Total Reward = 33.0
Episode 60: Total Reward = 14.0
Episode 80: Total Reward = 54.0
Episode 100: Total Reward = 64.0
Episode 120: Total Reward = 21.0
Episode 140: Total Reward = 19.0
Episode 160: Total Reward = 108.0
Episode 180: Total Reward = 36.0
Episode 200: Total Reward = 148.0
Episode 220: Total Reward = 66.0
Episode 240: Total Reward = 157.0
Episode 260: Total Reward = 133.0
Episode 280: Total Reward = 24.0
Episode 300: Total Reward = 158.0
Episode 320: Total Reward = 43.0
Episode 340: Total Reward = 433.0
Episode 360: Total Reward = 318.0
Episode 380: Total Reward = 326.0
Episode 400: Total Reward = 500.0
Episode 420: Total Reward = 152.0
Episode 440: Total Reward = 154.0
Episode 460: Total Reward = 66.0
Episode 480: Total Reward = 131.0


# 📚 Chapter 18 - Reinforcement Learning (RL)

---

## 🔸 1. Apa Itu Reinforcement Learning (RL)?
Reinforcement Learning → algoritma pembelajaran mesin di mana **agen belajar dengan trial & error** untuk mencapai tujuan dengan memaksimalkan reward.

**Lingkup RL:**
- **Agen** → model AI yang belajar
- **Lingkungan** → tempat agen berinteraksi
- **Aksi (Action)** → pilihan agen
- **Reward** → nilai feedback dari lingkungan
- **Policy** → strategi agen memilih aksi

---

## 🔸 2. Markov Decision Process (MDP)
Kerangka formal RL → model keputusan di mana **state berikutnya hanya bergantung pada state sekarang & action sekarang**.

| Komponen | Keterangan                |
| -------- | ------------------------- |
| S        | Set state                 |
| A        | Set aksi                  |
| P        | Probabilitas transisi     |
| R        | Fungsi reward             |
| γ        | Discount factor (0-1)     |

---

## 🔸 3. Goal dalam RL
Mencari **policy optimal** → memaksimalkan reward jangka panjang (expected cumulative reward).

**Formulasi:**
\[
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ...
\]

---

## 🔸 4. Value Function & Q-Function
- **V(s)** → *expected reward* dari state `s`
- **Q(s, a)** → *expected reward* dari state `s` jika melakukan aksi `a`

**Bellman Equation:**
\[
Q(s, a) = R(s, a) + \gamma \sum P(s' | s, a) \cdot \max_{a'} Q(s', a')
\]

---

## 🔸 5. Deep Q-Learning
✅ Menggunakan **Neural Network** → mendekati **Q-Function**.
- Input → State
- Output → Q-Value untuk semua aksi
- Loss → Mean Squared Error (MSE) antara prediksi Q dan target Q

---

## 🔸 6. ε-Greedy Policy
- **Eksplorasi (explore)** → mencoba aksi acak → peluang ε
- **Eksploitasi (exploit)** → pilih aksi dengan Q tertinggi → peluang (1-ε)

---

## 🔸 7. Replay Buffer
Menyimpan pengalaman → training dilakukan dengan sampling acak → **mengurangi korelasi antar pengalaman** → stabilisasi learning.

---

## 🔸 8. Variasi Metode RL
| Metode              | Keterangan                                |
| ------------------- | ----------------------------------------- |
| **Policy Gradients**| Model langsung belajar distribusi policy  |
| **Actor-Critic**    | Kombinasi Value Function + Policy Gradients |
| **DQN Variants**    | Double DQN, Dueling DQN, Prioritized Replay |

---

## ✅ Kesimpulan
- ✅ Reinforcement Learning → powerful untuk masalah *sequential decision making*
- ✅ Deep Q-Learning → standar modern → mendukung input state yang kompleks
- ✅ Untuk masalah lebih besar → lanjutkan ke **Policy Gradients & Actor-Critic**

